# Used Car Price Prediction & Market Analytics using PySpark

**Objective:** analyse a large used-car listing dataset with Apache Spark and predict a fair resale price from everything a buyer cares about: age, kilometres, fuel, owners, accident history, working condition and damaged parts.

**Pipeline:** `used_cars.csv` -> PySpark ETL -> Parquet -> Spark SQL EDA -> Spark MLlib models -> saved model + `model_metadata.json` -> `app.py` (Streamlit)

**How to run**
1. Put `used_cars.csv` next to this notebook (in Colab: upload it to the session files).
2. Run all cells top to bottom. Use `SAMPLE_FRACTION = 0.2` in the Settings cell for a quick test run.
3. Then start the app: `streamlit run app.py`

Notes
- Needs Java 11 or 17 (Colab already has it).
- **Windows only:** Spark needs two helper files to save models/Parquet. Create `C:\hadoop\bin`, put `winutils.exe` and `hadoop.dll` (Hadoop 3.3.x build, e.g. from the `cdarlint/winutils` repo, folder `hadoop-3.3.5/bin`) in it, then **restart the kernel**. The notebook finds them automatically (or set the `HADOOP_HOME` environment variable). Colab or WSL needs none of this.
- Nothing is hardcoded: the current year comes from the system clock, all tunables live in the Settings cell, and every list/range used by the app is derived from the data.

In [ ]:
import sys, subprocess, importlib

def _ensure(pkg, mod=None):
    try:
        importlib.import_module(mod or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg in ["pyspark", "pandas", "numpy", "matplotlib", "seaborn", "pyarrow", "psutil"]:
    _ensure(_pkg)

import os, re, json, time, shutil, datetime as dt
from contextlib import contextmanager
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
from IPython.display import display

# PySpark 3.5 predates pandas 3 and numpy 2; toPandas()/createDataFrame() can break with them
_maj = lambda v: int(v.split(".")[0])
if _maj(pd.__version__) >= 3 or _maj(np.__version__) >= 2:
    print(f"WARNING: pandas {pd.__version__} / numpy {np.__version__} are too new for PySpark 3.5.")
    print('Fix: run   pip install "pandas==2.2.3" "numpy==1.26.4"   then restart the kernel and run all cells again.')

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Settings
Every tunable value lives here. The simulated-condition probabilities and the price deductions are **assumptions** (the source data has no condition columns), so they are kept in one place and documented.

In [ ]:
CURRENT_YEAR = dt.date.today().year          # taken from the system clock, never typed in
RUN_STARTED = dt.datetime.now(dt.timezone.utc)

SETTINGS = {
    "INPUT_CSV": "used_cars.csv",
    "SEED": 42,
    "SAMPLE_FRACTION": 1.0,            # 1.0 = all rows. Use 0.2 for a quick test run
    "MAX_CAR_AGE": 40,                 # rows older than this are treated as invalid
    "SPLIT": [0.70, 0.15, 0.15],       # train / validation / test
    "OUTLIER_IQR_MULT": 1.5,           # IQR rule on log(price) and km_driven
    "BRAND_TIER_QUANTILES": [0.25, 0.50, 0.75],
    "BRAND_TIER_LABELS": ["Economy", "Mid", "Premium", "Luxury"],
    "GRADE_QUANTILES": [0.10, 0.35, 0.80],
    "GRADE_LABELS": ["Poor", "Fair", "Good", "Excellent"],
    "GRADE_RULE": {"w_working": 0.50, "w_damage": 0.50, "w_accident": 0.10},
    "DEAL_TOLERANCE": 0.08,            # asking price within +/- 8% of prediction = "Fair"

    # importance of each part / component (used for damage_score and working_score)
    "PART_WEIGHTS": {"seat": 1.0, "headlight": 1.0, "taillight": 0.8, "bumper": 1.2, "windshield": 1.2,
                     "door_dent": 1.0, "paint": 0.8, "tyre_wear": 0.8, "dashboard": 1.0},
    "COMPONENT_WEIGHTS": {"engine_ok": 3.0, "gearbox_ok": 2.5, "brakes_ok": 2.0, "suspension_ok": 1.5,
                          "battery_ok": 1.0, "ac_working": 1.0, "music_system_ok": 0.5, "power_windows_ok": 0.5},

    # SIMULATED condition generator (probabilities)
    "SIM": {
        "acc_base": 0.04, "acc_per_year": 0.012, "acc_per_100k_km": 0.03, "acc_max": 0.60,
        "acc_second": 0.25, "acc_third": 0.08,
        "ins_base": 0.97, "ins_per_year": 0.03, "ins_min": 0.50,
        "svc_base": 0.90, "svc_per_year": 0.02, "svc_min": 0.40,
        "tamper_base": 0.01, "tamper_per_100k_km": 0.02, "tamper_max": 0.15,
        "fail_per_year": 0.004, "fail_per_100k_km": 0.02, "fail_max": 0.50,
        "fail_base": {"engine_ok": 0.01, "gearbox_ok": 0.01, "brakes_ok": 0.02, "suspension_ok": 0.03,
                      "battery_ok": 0.04, "ac_working": 0.04, "music_system_ok": 0.05, "power_windows_ok": 0.03},
        "major_base": 0.02, "major_if_accident": 0.30,
        "minor_base": 0.08, "minor_per_year": 0.01, "minor_if_accident": 0.30,
    },

    # SIMULATED price deductions (share of the listing price)
    "DEDUCTIONS": {
        "accident_first": 0.10, "accident_extra": 0.03,
        "major_damage": 0.03, "minor_damage": 0.008,         # multiplied by the part weight
        "fault": {"engine_ok": 0.08, "gearbox_ok": 0.06, "brakes_ok": 0.03, "suspension_ok": 0.03,
                  "battery_ok": 0.01, "ac_working": 0.02, "music_system_ok": 0.005, "power_windows_ok": 0.01},
        "tamper": 0.08, "no_insurance": 0.02, "no_service": 0.03,
        "cap": 0.60, "noise_sigma": 0.02,
    },

    "MODEL_PARAMS": {
        "LinearRegression": {"maxIter": 100, "regParam": 0.0, "elasticNetParam": 0.0},
        "RandomForest": {"numTrees": 60, "maxDepth": 12},
        "GBT": {"maxIter": 60, "maxDepth": 6, "stepSize": 0.1},
    },
    "GRIDS": {
        "LinearRegression": {"regParam": [0.0, 0.01, 0.1]},
        "RandomForest": {"numTrees": [40, 80], "maxDepth": [10, 14]},
        "GBT": {"maxDepth": [5, 7], "maxIter": [40, 80]},
    },
    "CV_ENABLED": True, "CV_FOLDS": 3, "CV_SAMPLE_FRACTION": 0.3,
    "KMEANS_K_RANGE": [3, 7], "KMEANS_SAMPLE_ROWS": 50000,
    "EDA_SAMPLE_ROWS": 100000,
}

SEED = SETTINGS["SEED"]
PARTS = list(SETTINGS["PART_WEIGHTS"])
COMPONENTS = list(SETTINGS["COMPONENT_WEIGHTS"])

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
for _d in (DATA_DIR, MODELS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

TIMINGS = {}

@contextmanager
def timed(name):
    t0 = time.time()
    yield
    TIMINGS[name] = round(time.time() - t0, 1)
    print(f"[{name}] {TIMINGS[name]}s")

print("Current year (runtime):", CURRENT_YEAR)

## 2. Spark session
Parallelism and memory are set from the CPU cores and RAM detected at runtime.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Windows only: Spark needs winutils.exe + hadoop.dll to save models/Parquet (HADOOP_HOME\bin)
if os.name == "nt":
    _hh = Path(os.environ.get("HADOOP_HOME", r"C:\hadoop"))
    if (_hh / "bin" / "winutils.exe").exists() and (_hh / "bin" / "hadoop.dll").exists():
        os.environ["HADOOP_HOME"] = str(_hh)
        os.environ["PATH"] = str(_hh / "bin") + os.pathsep + os.environ["PATH"]
        print("Windows: using Hadoop native files from", _hh)
    else:
        print("WARNING (Windows): winutils.exe and/or hadoop.dll not found in", _hh / "bin",
              "- saving the model will fail. Follow the Windows setup note at the top of the notebook, then restart the kernel.")

if shutil.which("java") is None:
    print("WARNING: Java not found. Install Java 11 or 17, restart the kernel and run again.")

CORES = os.cpu_count() or 2
RAM_GB = psutil.virtual_memory().total / 1e9
DRIVER_MEM_GB = max(2, int(RAM_GB * 0.6))
SHUFFLE_PARTS = CORES * 2

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

spark = (SparkSession.builder.master(f"local[{CORES}]").appName("UsedCarBDA")
         .config("spark.driver.memory", f"{DRIVER_MEM_GB}g")
         .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTS))
         .config("spark.default.parallelism", str(SHUFFLE_PARTS))
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark {spark.version} | cores={CORES} | driver memory={DRIVER_MEM_GB}g | shuffle partitions={SHUFFLE_PARTS}")
print("Spark UI:", spark.sparkContext.uiWebUrl)

## 3. Load the data
Column names are matched case-insensitively with common synonyms, so a different used-car CSV also works. The mapping that was used is printed.

In [ ]:
SYNONYMS = {
    "name": ["name", "car_name", "title"],
    "brand": ["brand", "make", "manufacturer", "company"],
    "model": ["model", "car_model"],
    "location": ["location", "city"],
    "year": ["year", "manufacturing_year", "mfg_year", "model_year", "registration_year"],
    "km_driven": ["kilometers_driven", "kilometres_driven", "km_driven", "kms_driven", "odometer"],
    "fuel": ["fuel_type", "fuel"],
    "transmission": ["transmission", "gearbox", "transmission_type"],
    "owner": ["owner_type", "owner", "owners", "ownership"],
    "power": ["max_power", "power", "power_bhp", "max_power_bhp"],
    "seats": ["seats", "seating_capacity", "no_of_seats"],
    "mileage": ["mileage", "mileage_kmpl", "kmpl", "fuel_efficiency"],
    "engine": ["engine", "engine_cc", "cc", "engine_capacity"],
    "price": ["selling_price", "price", "selling_price_inr", "sale_price", "resale_price"],
}
REQUIRED = ["location", "year", "km_driven", "fuel", "transmission", "owner", "power", "seats", "mileage", "engine", "price"]

def norm(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).strip().lower()).strip("_")

csv_path = Path(SETTINGS["INPUT_CSV"])
if not csv_path.exists():
    raise FileNotFoundError(f"Could not find {csv_path.resolve()}. Put the CSV next to this notebook or change INPUT_CSV in Settings.")

header = pd.read_csv(csv_path, nrows=0).columns.tolist()
mapping = {}
for canon, alts in SYNONYMS.items():
    for col in header:
        if norm(col) in alts and col not in mapping.values():
            mapping[canon] = col
            break

missing = [c for c in REQUIRED if c not in mapping]
if "brand" not in mapping and "name" not in mapping:
    missing.append("brand (or name)")
if "model" not in mapping and "name" not in mapping:
    missing.append("model (or name)")
if missing:
    raise ValueError(f"Required columns not found in the CSV: {missing}. Columns present: {header}")

print("Column mapping used (standard name <- CSV column):")
for k, v in mapping.items():
    print(f"  {k:13s} <- {v}")

TYPES = {"year": IntegerType(), "km_driven": DoubleType(), "power": DoubleType(), "seats": DoubleType(),
         "mileage": DoubleType(), "engine": DoubleType(), "price": DoubleType()}
inverse = {v: k for k, v in mapping.items()}
schema = StructType([StructField(inverse.get(c, c), TYPES.get(inverse.get(c), StringType()), True) for c in header])

raw = spark.read.csv(str(csv_path), header=True, schema=schema)
if SETTINGS["SAMPLE_FRACTION"] < 1.0:
    raw = raw.sample(False, SETTINGS["SAMPLE_FRACTION"], SEED)
if "brand" not in raw.columns:
    raw = raw.withColumn("brand", F.split(F.col("name"), " ").getItem(0))
if "model" not in raw.columns:
    raw = raw.withColumn("model", F.split(F.col("name"), " ").getItem(1))

raw = raw.cache()
N_RAW = raw.count()
print(f"\nRows loaded: {N_RAW:,}")
raw.printSchema()
display(raw.limit(5).toPandas())

## 4. Data quality report

In [ ]:
nulls = raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw.columns]).toPandas().T
nulls.columns = ["null_count"]
nulls["null_pct"] = (nulls["null_count"] / N_RAW * 100).round(2)
n_dups = N_RAW - raw.dropDuplicates().count()

print(f"Rows: {N_RAW:,} | exact duplicate rows: {n_dups:,}")
display(nulls[nulls["null_count"] > 0] if (nulls["null_count"] > 0).any() else nulls)

num_cols_raw = ["year", "km_driven", "power", "seats", "mileage", "engine", "price"]
display(raw.select(num_cols_raw).describe().toPandas())

## 5. Cleaning and feature engineering (PySpark)
- Missing power / mileage / engine are filled with the **median of the same brand + model + fuel**; missing seats with the **most common value for that model**.
- Outliers are removed with the IQR rule on `log(price)` (so expensive luxury cars are not wrongly removed) and on `km_driven`.
- New features: `car_age`, `km_per_year`, `power_per_cc`, and `brand_tier` (from each brand's median price quantiles).

In [ ]:
def impute_numeric_by_group(d, col, group_cols):
    med = d.groupBy(*group_cols).agg(F.percentile_approx(col, 0.5).alias("_grp"))
    glob = d.agg(F.percentile_approx(col, 0.5)).first()[0]
    d = d.join(med, list(group_cols), "left")
    return d.withColumn(col, F.coalesce(F.col(col), F.col("_grp"), F.lit(glob))).drop("_grp")

def iqr_bounds(d, col, mult):
    q1, q3 = d.approxQuantile(col, [0.25, 0.75], 0.001)
    iqr = q3 - q1
    return q1 - mult * iqr, q3 + mult * iqr

with timed("cleaning"):
    df = raw
    for c in ["brand", "model", "location", "fuel", "transmission", "owner"]:
        df = df.withColumn(c, F.trim(F.col(c)))

    df = df.dropna(subset=["brand", "model", "location", "fuel", "transmission", "owner", "year", "km_driven", "price"])
    df = df.filter((F.col("price") > 0) & (F.col("km_driven") >= 0) &
                   (F.col("year") <= CURRENT_YEAR) & (F.col("year") >= CURRENT_YEAR - SETTINGS["MAX_CAR_AGE"]))

    # impute numeric specs by brand+model+fuel median
    for c in ["power", "mileage", "engine"]:
        df = impute_numeric_by_group(df, c, ("brand", "model", "fuel"))

    # impute seats with the most common value for the model
    seat_counts = df.filter(F.col("seats").isNotNull()).groupBy("brand", "model", "seats").count()
    w = Window.partitionBy("brand", "model").orderBy(F.desc("count"), F.desc("seats"))
    seat_mode = (seat_counts.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1)
                 .select("brand", "model", F.col("seats").alias("_seat_mode")))
    glob_seat = seat_counts.groupBy("seats").agg(F.sum("count").alias("n")).orderBy(F.desc("n")).first()["seats"]
    df = (df.join(seat_mode, ["brand", "model"], "left")
            .withColumn("seats", F.coalesce(F.col("seats"), F.col("_seat_mode"), F.lit(glob_seat)))
            .drop("_seat_mode"))

    df = df.filter(F.col("engine") > 0).dropDuplicates()
    n_before_outliers = df.count()

    # IQR outlier removal
    df = df.withColumn("_log_price", F.log("price"))
    lo_p, hi_p = iqr_bounds(df, "_log_price", SETTINGS["OUTLIER_IQR_MULT"])
    _, hi_k = iqr_bounds(df, "km_driven", SETTINGS["OUTLIER_IQR_MULT"])
    df = df.filter(F.col("_log_price").between(lo_p, hi_p) & (F.col("km_driven") <= hi_k)).drop("_log_price")
    print(f"Price bounds from IQR: {np.exp(lo_p):,.0f} to {np.exp(hi_p):,.0f} | km upper bound: {hi_k:,.0f}")

    # derived features
    df = (df.withColumn("car_age", F.greatest(F.lit(CURRENT_YEAR) - F.col("year"), F.lit(0)))
            .withColumn("km_per_year", F.col("km_driven") / F.greatest(F.col("car_age"), F.lit(1)))
            .withColumn("power_per_cc", F.col("power") / F.col("engine")))

    # brand tier from per-brand median price quantiles
    brand_med = df.groupBy("brand").agg(F.percentile_approx("price", 0.5).alias("med_price")).toPandas()
    cuts = brand_med["med_price"].quantile(SETTINGS["BRAND_TIER_QUANTILES"]).tolist()
    tier_labels = SETTINGS["BRAND_TIER_LABELS"]
    assert len(tier_labels) == len(cuts) + 1, "BRAND_TIER_LABELS must have one more entry than BRAND_TIER_QUANTILES"
    brand_med["brand_tier"] = brand_med["med_price"].apply(lambda v: tier_labels[int(sum(v > c for c in cuts))])
    BRAND_TIER_MAP = dict(zip(brand_med["brand"], brand_med["brand_tier"]))
    df = df.join(spark.createDataFrame(brand_med[["brand", "brand_tier"]]), "brand", "left")

    df = df.cache()
    N_CLEAN = df.count()

print(f"Rows before cleaning: {N_RAW:,} | after cleaning: {N_CLEAN:,} (removed {N_RAW - N_CLEAN:,})")
display(brand_med.sort_values("med_price")[["brand", "med_price", "brand_tier"]].reset_index(drop=True))

In [ ]:
def safe_save_parquet(d, path):
    try:
        d.write.mode("overwrite").parquet(str(path))
        print("Saved:", path)
        return True
    except Exception as e:
        print("Could not write Parquet (continuing in memory). Reason:", str(e).splitlines()[0])
        return False

safe_save_parquet(df, DATA_DIR / "cars_clean.parquet")

## 6. SIMULATED buyer-condition features

> **Important:** the source CSV has no accident, damage or working-condition columns, so these features are **simulated** with documented probabilities (they depend on car age, kilometres and accidents). The price deductions below are **assumptions**, not market facts. Both are configurable in the Settings cell and must be stated as limitations in the report.

**Simulated pricing assumptions** (all values come from `SETTINGS["DEDUCTIONS"]`):
- First accident: a fixed share of the price, plus a smaller share for each extra accident.
- Each damaged part: minor or major deduction, scaled by the part's importance weight.
- Each faulty component: its own deduction (engine and gearbox are the largest).
- Odometer tampering, missing insurance and missing service history: fixed deductions.
- Total deduction is capped, and a small random noise is added.

`final_price = price x (1 - total_deduction) x noise` is the target the models learn.

In [ ]:
S = SETTINGS["SIM"]
D = SETTINGS["DEDUCTIONS"]
_counter = [0]

def rnd():
    _counter[0] += 1
    return F.rand(SEED + _counter[0])

def prob_expr(base, per_year=0.0, per_100k=0.0, lo=None, hi=None):
    p = F.lit(float(base)) + F.lit(float(per_year)) * F.col("car_age") + F.lit(float(per_100k)) * F.col("km_driven") / F.lit(1e5)
    if hi is not None:
        p = F.least(F.lit(float(hi)), p)
    if lo is not None:
        p = F.greatest(F.lit(float(lo)), p)
    return p

with timed("condition simulation"):
    sim = df

    # accidents
    sim = sim.withColumn("accident_history", (rnd() < prob_expr(S["acc_base"], S["acc_per_year"], S["acc_per_100k_km"], hi=S["acc_max"])).cast("int"))
    sim = sim.withColumn("accident_count",
                         F.when(F.col("accident_history") == 0, F.lit(0))
                          .otherwise(1 + (rnd() < S["acc_second"]).cast("int") + (rnd() < S["acc_third"]).cast("int")))

    # papers / history
    sim = sim.withColumn("insurance_valid", (rnd() < prob_expr(S["ins_base"], -S["ins_per_year"], lo=S["ins_min"])).cast("int"))
    sim = sim.withColumn("service_history_available", (rnd() < prob_expr(S["svc_base"], -S["svc_per_year"], lo=S["svc_min"])).cast("int"))
    sim = sim.withColumn("odometer_tamper_suspected", (rnd() < prob_expr(S["tamper_base"], 0.0, S["tamper_per_100k_km"], hi=S["tamper_max"])).cast("int"))

    # working components (1 = working)
    for comp in COMPONENTS:
        p_fail = prob_expr(S["fail_base"][comp], S["fail_per_year"], S["fail_per_100k_km"], hi=S["fail_max"])
        sim = sim.withColumn(comp, (rnd() >= p_fail).cast("int"))

    # damaged parts (0 none, 1 minor, 2 major)
    for part in PARTS:
        sim = sim.withColumn(f"_r_{part}", rnd())
        p_major = F.when(F.col("accident_history") == 1, F.lit(S["major_if_accident"])).otherwise(F.lit(S["major_base"]))
        p_minor = F.when(F.col("accident_history") == 1, F.lit(S["minor_if_accident"])) \
                   .otherwise(F.lit(S["minor_base"]) + F.lit(S["minor_per_year"]) * F.col("car_age"))
        sim = sim.withColumn(f"{part}_damage",
                             F.when(F.col(f"_r_{part}") < p_major, F.lit(2))
                              .when(F.col(f"_r_{part}") < p_major + p_minor, F.lit(1))
                              .otherwise(F.lit(0))).drop(f"_r_{part}")

    # scores
    pw, cw = SETTINGS["PART_WEIGHTS"], SETTINGS["COMPONENT_WEIGHTS"]
    DAMAGE_MAX = 2.0 * sum(pw.values())
    sim = sim.withColumn("damage_score", sum(F.col(f"{p}_damage") * F.lit(float(w)) for p, w in pw.items()))
    sim = sim.withColumn("working_score", sum(F.col(c) * F.lit(float(w)) for c, w in cw.items()) / F.lit(float(sum(cw.values()))))
    GR = SETTINGS["GRADE_RULE"]
    sim = sim.withColumn("condition_index",
                         F.lit(GR["w_working"]) * F.col("working_score")
                         + F.lit(GR["w_damage"]) * (F.lit(1.0) - F.col("damage_score") / F.lit(DAMAGE_MAX))
                         - F.lit(GR["w_accident"]) * F.col("accident_history"))

    # condition grade from data quantiles
    GRADE_CUTOFFS = sim.approxQuantile("condition_index", SETTINGS["GRADE_QUANTILES"], 0.001)
    glabels = SETTINGS["GRADE_LABELS"]
    assert len(glabels) == len(GRADE_CUTOFFS) + 1, "GRADE_LABELS must have one more entry than GRADE_QUANTILES"
    grade = F.when(F.col("condition_index") < GRADE_CUTOFFS[0], glabels[0])
    for i in range(1, len(GRADE_CUTOFFS)):
        grade = grade.when(F.col("condition_index") < GRADE_CUTOFFS[i], glabels[i])
    sim = sim.withColumn("condition_grade", grade.otherwise(glabels[-1]))

    # price deductions -> final_price (the model target)
    ded = F.when(F.col("accident_history") == 1,
                 F.lit(D["accident_first"]) + F.lit(D["accident_extra"]) * (F.col("accident_count") - 1)).otherwise(F.lit(0.0))
    for part, w in pw.items():
        ded = ded + (F.when(F.col(f"{part}_damage") == 2, F.lit(D["major_damage"] * w))
                      .when(F.col(f"{part}_damage") == 1, F.lit(D["minor_damage"] * w))
                      .otherwise(F.lit(0.0)))
    for comp, v in D["fault"].items():
        ded = ded + F.when(F.col(comp) == 0, F.lit(v)).otherwise(F.lit(0.0))
    ded = ded + F.col("odometer_tamper_suspected") * F.lit(D["tamper"]) \
              + (1 - F.col("insurance_valid")) * F.lit(D["no_insurance"]) \
              + (1 - F.col("service_history_available")) * F.lit(D["no_service"])
    sim = sim.withColumn("total_deduction", F.least(F.lit(D["cap"]), ded))
    sim = sim.withColumn("final_price", F.round(F.col("price") * (1 - F.col("total_deduction")) * F.exp(F.randn(SEED + 999) * F.lit(D["noise_sigma"]))))
    sim = sim.withColumn("price_ratio", F.col("final_price") / F.col("price"))

    safe_save_parquet(sim, DATA_DIR / "cars_enriched.parquet")
    df = sim.cache()
    N_ENRICHED = df.count()

print(f"Enriched rows: {N_ENRICHED:,}")
print("Condition grade cut-offs (from data quantiles):", [round(c, 4) for c in GRADE_CUTOFFS])
display(df.select("brand", "model", "car_age", "km_driven", "accident_history", "damage_score", "working_score",
                  "condition_grade", "price", "total_deduction", "final_price").limit(8).toPandas())

## 7. Exploratory analysis with Spark SQL
Each chart is followed by a one-line insight computed from the data.

In [ ]:
df.createOrReplaceTempView("cars")

def q(sql):
    return spark.sql(sql).toPandas()

def inr(v):
    return f"Rs {v:,.0f}"

def insight_rank(p, lab, val, fmt=inr):
    hi, lo = p.loc[p[val].idxmax()], p.loc[p[val].idxmin()]
    print(f"Insight: highest = {hi[lab]} ({fmt(hi[val])}), lowest = {lo[lab]} ({fmt(lo[val])}).")

# --- 1-5: average price by brand / fuel / transmission / owner / location
for col, title in [("brand", "brand"), ("fuel", "fuel type"), ("transmission", "transmission"),
                   ("owner", "owner type"), ("location", "city")]:
    p = q(f"SELECT {col} AS grp, ROUND(AVG(final_price)) AS avg_price, COUNT(*) AS listings "
          f"FROM cars GROUP BY {col} ORDER BY avg_price DESC")
    fig, ax = plt.subplots(figsize=(9, max(3, 0.35 * len(p))))
    sns.barplot(data=p, y="grp", x="avg_price", ax=ax, color="#3b7dd8")
    ax.set_title(f"Average price by {title}")
    ax.set_xlabel("Average price (INR)")
    ax.set_ylabel(title.capitalize())
    plt.tight_layout()
    plt.show()
    insight_rank(p, "grp", "avg_price")

In [ ]:
# --- 6: depreciation curve
p = q("SELECT car_age, AVG(price) AS listing_price, AVG(final_price) AS final_price, COUNT(*) AS n "
      "FROM cars GROUP BY car_age ORDER BY car_age")
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(p["car_age"], p["listing_price"], marker="o", label="Listing price")
ax.plot(p["car_age"], p["final_price"], marker="o", label="Final price after condition")
ax.set_title("Depreciation curve: average price vs car age")
ax.set_xlabel("Car age (years)")
ax.set_ylabel("Average price (INR)")
ax.legend()
plt.tight_layout()
plt.show()
a0 = int(p["car_age"].min())
a1 = int(min(a0 + 5, p["car_age"].max()))
v0 = float(p.loc[p.car_age == a0, "final_price"].iloc[0])
v1 = float(p.loc[p.car_age == a1, "final_price"].iloc[0])
print(f"Insight: average price falls {100 * (1 - v1 / v0):.1f}% between age {a0} and age {a1}.")

# --- 7: accident effect
p = q("SELECT accident_history, ROUND(AVG(final_price)) AS avg_final_price, ROUND(AVG(price_ratio), 4) AS avg_ratio, COUNT(*) AS n "
      "FROM cars GROUP BY accident_history ORDER BY accident_history")
p["accident"] = p["accident_history"].map({0: "No accident", 1: "Accident"})
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=p, x="accident", y="avg_ratio", ax=ax, color="#d8643b")
ax.set_title("Effect of accident history on price (final / listing)")
ax.set_xlabel("")
ax.set_ylabel("Average price ratio")
plt.tight_layout()
plt.show()
r = p.set_index("accident_history")["avg_ratio"]
print(f"Insight: cars with an accident sell for {100 * (1 - r[1] / r[0]):.1f}% less than accident-free ones, on average.")

In [ ]:
# --- 8: effect of each damaged part (major damage vs none)
rows = []
for part in PARTS:
    t = q(f"SELECT {part}_damage AS lvl, AVG(price_ratio) AS r FROM cars GROUP BY {part}_damage")
    for lvl, name in [(0, "None"), (1, "Minor"), (2, "Major")]:
        v = t.loc[t.lvl == lvl, "r"]
        rows.append({"part": part, "level": name, "avg_ratio": float(v.iloc[0]) if len(v) else np.nan})
pp = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=pp, x="part", y="avg_ratio", hue="level", hue_order=["None", "Minor", "Major"], ax=ax)
ax.set_title("Effect of each damaged part on price (final / listing)")
ax.set_xlabel("Part")
ax.set_ylabel("Average price ratio")
ax.set_ylim(max(0, pp["avg_ratio"].min() - 0.05), 1.0)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
maj = pp[pp.level == "Major"].dropna().sort_values("avg_ratio")
print(f"Insight: major damage hurts the price most for '{maj.iloc[0]['part']}' (ratio {maj.iloc[0]['avg_ratio']:.3f}) and least for '{maj.iloc[-1]['part']}' ({maj.iloc[-1]['avg_ratio']:.3f}).")

# --- 9: working score
p = q("SELECT ROUND(working_score, 1) AS ws, AVG(price_ratio) AS avg_ratio, COUNT(*) AS n FROM cars GROUP BY ROUND(working_score, 1) ORDER BY ws")
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p["ws"], p["avg_ratio"], marker="o")
ax.set_title("Effect of working condition on price")
ax.set_xlabel("Working score (1.0 = everything works)")
ax.set_ylabel("Average price ratio")
plt.tight_layout()
plt.show()
print(f"Insight: cars where everything works keep {100 * p['avg_ratio'].iloc[-1]:.1f}% of the listing price versus {100 * p['avg_ratio'].iloc[0]:.1f}% for the worst working score.")

In [ ]:
# --- 10: top 10 models by listings
p = q("SELECT CONCAT(brand, ' ', model) AS car, COUNT(*) AS listings, ROUND(AVG(final_price)) AS avg_price "
      "FROM cars GROUP BY brand, model ORDER BY listings DESC LIMIT 10")
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=p, y="car", x="listings", ax=ax, color="#4aa36c")
ax.set_title("Top 10 models by number of listings")
ax.set_xlabel("Listings")
ax.set_ylabel("Model")
plt.tight_layout()
plt.show()
print(f"Insight: '{p.iloc[0]['car']}' is the most listed model ({int(p.iloc[0]['listings']):,} listings, average {inr(p.iloc[0]['avg_price'])}).")

# --- 11 & 12: distribution and correlation (on a sample)
n_eda = df.count()
frac = min(1.0, SETTINGS["EDA_SAMPLE_ROWS"] / max(n_eda, 1))
eda_pdf = df.select("km_per_year", "car_age", "km_driven", "engine", "power", "mileage", "seats", "damage_score",
                    "working_score", "accident_history", "price", "final_price").sample(False, frac, SEED).toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(eda_pdf["km_per_year"], bins=60, ax=ax, color="#8a5fd1")
ax.set_title("Distribution of kilometres driven per year")
ax.set_xlabel("km per year")
ax.set_ylabel("Cars")
plt.tight_layout()
plt.show()
print(f"Insight: the median car is driven {eda_pdf['km_per_year'].median():,.0f} km per year.")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(eda_pdf.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, annot_kws={"size": 7})
ax.set_title("Correlation heatmap of numeric features")
plt.tight_layout()
plt.show()
corr_price = eda_pdf.corr(numeric_only=True)["final_price"].drop(["final_price", "price"]).abs().sort_values(ascending=False)
print(f"Insight: the numeric feature most correlated with final price is '{corr_price.index[0]}' (|r| = {corr_price.iloc[0]:.2f}).")

In [ ]:
# --- 13: condition grade distribution
p = q("SELECT condition_grade, COUNT(*) AS listings, ROUND(AVG(price_ratio), 4) AS avg_ratio FROM cars GROUP BY condition_grade")
order = SETTINGS["GRADE_LABELS"]
p["condition_grade"] = pd.Categorical(p["condition_grade"], categories=order, ordered=True)
p = p.sort_values("condition_grade")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=p, x="condition_grade", y="listings", ax=axes[0], color="#3b7dd8")
axes[0].set_title("Condition grade distribution")
axes[0].set_xlabel("Condition grade")
axes[0].set_ylabel("Listings")
sns.barplot(data=p, x="condition_grade", y="avg_ratio", ax=axes[1], color="#d8643b")
axes[1].set_title("Average price ratio by condition grade")
axes[1].set_xlabel("Condition grade")
axes[1].set_ylabel("Final / listing price")
plt.tight_layout()
plt.show()
print(f"Insight: '{order[0]}' cars keep {100 * p.iloc[0]['avg_ratio']:.1f}% of the listing price, '{order[-1]}' cars keep {100 * p.iloc[-1]['avg_ratio']:.1f}%.")

# --- 14: resale value by fuel type over age
p = q("SELECT car_age, fuel, AVG(final_price) AS avg_price FROM cars GROUP BY car_age, fuel ORDER BY car_age")
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(data=p, x="car_age", y="avg_price", hue="fuel", marker="o", ax=ax)
ax.set_title("Resale value by fuel type over car age")
ax.set_xlabel("Car age (years)")
ax.set_ylabel("Average price (INR)")
plt.tight_layout()
plt.show()
ret = {}
for f_, g in p.groupby("fuel"):
    g = g.sort_values("car_age")
    ret[f_] = g["avg_price"].iloc[min(5, len(g) - 1)] / g["avg_price"].iloc[0]
best_f = max(ret, key=ret.get)
print(f"Insight: after about 5 years, '{best_f}' cars retain the most value ({100 * ret[best_f]:.0f}% of the newest cars' average price).")

# --- 15: insurance and service history
p = q("SELECT insurance_valid, service_history_available, ROUND(AVG(price_ratio), 4) AS avg_ratio, COUNT(*) AS n "
      "FROM cars GROUP BY insurance_valid, service_history_available")
p["group"] = p.apply(lambda r: f"Insurance {'yes' if r.insurance_valid else 'no'} / Service {'yes' if r.service_history_available else 'no'}", axis=1)
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=p, y="group", x="avg_ratio", ax=ax, color="#4aa36c")
ax.set_title("Impact of insurance and service history on price")
ax.set_xlabel("Average price ratio (final / listing)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()
insight_rank(p, "group", "avg_ratio", fmt=lambda v: f"{100 * v:.1f}%")

## 8. Modelling with Spark MLlib
- Target: `log(final_price)` (prices are multiplicative, so this suits linear models and trees alike). All reported errors are converted back to rupees.
- Data is split into train / validation / test. The best model type is chosen by **validation RMSE**, then tuned with a small cross-validated grid.
- An **ablation** retrains the best model without the condition features to show how much they matter.

In [ ]:
CAT_COLS = ["brand", "model", "location", "fuel", "transmission", "owner", "brand_tier"]
BASE_NUM = ["car_age", "km_driven", "km_per_year", "engine", "power", "seats", "mileage", "power_per_cc"]
COND_NUM = (["accident_history", "accident_count", "insurance_valid", "service_history_available", "odometer_tamper_suspected"]
            + COMPONENTS + [f"{p}_damage" for p in PARTS] + ["damage_score", "working_score"])
ALL_INPUTS = CAT_COLS + BASE_NUM + COND_NUM

ml = (df.withColumn("log_price", F.log("final_price"))
        .select(ALL_INPUTS + ["final_price", "price", "log_price", "condition_grade"]).cache())
train, valid, test = ml.randomSplit(SETTINGS["SPLIT"], seed=SEED)
print(f"train={train.count():,}  validation={valid.count():,}  test={test.count():,}")

def make_pre(cat_cols, num_cols, scale=False):
    idx = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
    enc = OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_cols], outputCols=[f"{c}_ohe" for c in cat_cols], handleInvalid="keep")
    asm = VectorAssembler(inputCols=[f"{c}_ohe" for c in cat_cols] + list(num_cols),
                          outputCol="features_raw" if scale else "features", handleInvalid="keep")
    stages = idx + [enc, asm]
    if scale:
        stages.append(StandardScaler(inputCol="features_raw", outputCol="features", withMean=False, withStd=True))
    return stages

def make_estimator(name, **override):
    p = dict(SETTINGS["MODEL_PARAMS"][name])
    p.update(override)
    if name == "LinearRegression":
        return LinearRegression(featuresCol="features", labelCol="log_price", **p)
    if name == "RandomForest":
        return RandomForestRegressor(featuresCol="features", labelCol="log_price", seed=SEED, **p)
    if name == "GBT":
        return GBTRegressor(featuresCol="features", labelCol="log_price", seed=SEED, **p)
    raise ValueError(name)

def evaluate(model, data):
    pred = model.transform(data).withColumn("pred_price", F.exp(F.col("prediction")))
    out = {m: RegressionEvaluator(labelCol="final_price", predictionCol="pred_price", metricName=m).evaluate(pred)
           for m in ["rmse", "mae", "r2"]}
    return out, pred

CANDIDATES = ["LinearRegression", "RandomForest", "GBT"]
fitted, scores = {}, {}
for name in CANDIDATES:
    with timed(f"fit {name}"):
        pipe = Pipeline(stages=make_pre(CAT_COLS, BASE_NUM + COND_NUM, scale=(name == "LinearRegression")) + [make_estimator(name)])
        fitted[name] = pipe.fit(train)
    scores[name] = {"validation": evaluate(fitted[name], valid)[0], "test": evaluate(fitted[name], test)[0]}

summary = pd.DataFrame({n: {"val_RMSE": s["validation"]["rmse"], "test_RMSE": s["test"]["rmse"],
                            "test_MAE": s["test"]["mae"], "test_R2": s["test"]["r2"]} for n, s in scores.items()}).T
display(summary.round(3))
BEST_NAME = min(scores, key=lambda n: scores[n]["validation"]["rmse"])
print("Best model by validation RMSE:", BEST_NAME)

In [ ]:
# Hyper-parameter tuning of the best model (cross-validated on a sample of the training data)
best_params = dict(SETTINGS["MODEL_PARAMS"][BEST_NAME])
scale_flag = (BEST_NAME == "LinearRegression")
pre_stages = make_pre(CAT_COLS, BASE_NUM + COND_NUM, scale=scale_flag)

with timed(f"tune {BEST_NAME}"):
    if SETTINGS["CV_ENABLED"]:
        est = make_estimator(BEST_NAME)
        builder = ParamGridBuilder()
        for pname, values in SETTINGS["GRIDS"][BEST_NAME].items():
            builder = builder.addGrid(getattr(est, pname), values)
        grid = builder.build()
        cv = CrossValidator(estimator=Pipeline(stages=pre_stages + [est]), estimatorParamMaps=grid,
                            evaluator=RegressionEvaluator(labelCol="log_price", predictionCol="prediction", metricName="rmse"),
                            numFolds=SETTINGS["CV_FOLDS"], seed=SEED, parallelism=2)
        cv_sample = train.sample(False, SETTINGS["CV_SAMPLE_FRACTION"], SEED)
        cv_model = cv.fit(cv_sample)
        best_pm = grid[int(np.argmin(cv_model.avgMetrics))]
        best_params.update({p.name: v for p, v in best_pm.items()})
        cv_table = pd.DataFrame([{**{p.name: v for p, v in pm.items()}, "cv_rmse_log": m} for pm, m in zip(grid, cv_model.avgMetrics)])
        display(cv_table.sort_values("cv_rmse_log").round(4))
        final_est = est.copy(best_pm)
    else:
        final_est = make_estimator(BEST_NAME)

    FINAL_MODEL = Pipeline(stages=pre_stages + [final_est]).fit(train)

final_val, _ = evaluate(FINAL_MODEL, valid)
final_test, test_pred = evaluate(FINAL_MODEL, test)
scores[f"{BEST_NAME} (tuned)"] = {"validation": final_val, "test": final_test}
print("Best parameters:", best_params)
print(f"Tuned {BEST_NAME}: validation RMSE = {final_val['rmse']:,.0f} | test RMSE = {final_test['rmse']:,.0f} | test MAE = {final_test['mae']:,.0f} | test R2 = {final_test['r2']:.4f}")

In [ ]:
# Ablation: same model WITHOUT the condition features
with timed("ablation"):
    abl_est = make_estimator(BEST_NAME, **{k: v for k, v in best_params.items() if k in SETTINGS["MODEL_PARAMS"][BEST_NAME]})
    abl_model = Pipeline(stages=make_pre(CAT_COLS, BASE_NUM, scale=scale_flag) + [abl_est]).fit(train)
abl_test, _ = evaluate(abl_model, test)

ablation = pd.DataFrame({"with condition features": final_test, "without condition features": abl_test}).T
display(ablation.round(3))
gain = 100 * (1 - final_test["rmse"] / abl_test["rmse"])
print(f"Insight: adding accident, damage and working-condition features lowers the test RMSE by {gain:.1f}% and raises R2 from {abl_test['r2']:.3f} to {final_test['r2']:.3f}.")

In [ ]:
# Classifier: predict the condition grade of a listing from its basic details only (before any inspection)
with timed("classifier"):
    rfc = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=60, maxDepth=10, seed=SEED)
    cls_pipe = Pipeline(stages=make_pre(CAT_COLS, BASE_NUM) +
                        [StringIndexer(inputCol="condition_grade", outputCol="label", handleInvalid="keep"), rfc])
    cls_model = cls_pipe.fit(train)
cls_pred = cls_model.transform(test)
acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(cls_pred)
f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1").evaluate(cls_pred)
print(f"Condition-grade classifier (basic details only): accuracy = {acc:.3f} | weighted F1 = {f1:.3f}")
print("Note: the grade itself is computed from inspection details; this model shows how well it can be guessed from age, km, fuel, owner etc. alone.")

In [ ]:
# K-Means market segments (k chosen by silhouette score)
KM_FEATURES = ["car_age", "km_driven", "engine", "power", "log_price"]
n_km = ml.count()
km_df = ml.sample(False, min(1.0, SETTINGS["KMEANS_SAMPLE_ROWS"] / max(n_km, 1)), SEED)
prep = Pipeline(stages=[VectorAssembler(inputCols=KM_FEATURES, outputCol="raw"),
                        StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True)]).fit(km_df)
km_data = prep.transform(km_df).cache()

k_lo, k_hi = SETTINGS["KMEANS_K_RANGE"]
silhouette = {}
with timed("kmeans"):
    for k in range(k_lo, k_hi + 1):
        m = KMeans(k=k, seed=SEED, featuresCol="features", maxIter=30).fit(km_data)
        silhouette[k] = ClusteringEvaluator(featuresCol="features", metricName="silhouette",
                                            distanceMeasure="squaredEuclidean").evaluate(m.transform(km_data))
BEST_K = max(silhouette, key=silhouette.get)
print("Silhouette by k:", {k: round(v, 3) for k, v in silhouette.items()}, "-> best k =", BEST_K)

km_model = KMeans(k=BEST_K, seed=SEED, featuresCol="features", maxIter=30).fit(km_data)
seg = (km_model.transform(km_data).groupBy("prediction")
       .agg(F.count("*").alias("cars"), F.round(F.avg("car_age"), 1).alias("avg_age"),
            F.round(F.avg("km_driven"), 0).alias("avg_km"), F.round(F.avg("engine"), 0).alias("avg_engine_cc"),
            F.round(F.avg("power"), 1).alias("avg_power_bhp"), F.round(F.avg("final_price"), 0).alias("avg_price"))
       .orderBy("avg_price").toPandas())
seg = seg.rename(columns={"prediction": "segment"})
display(seg)
print(f"Insight: the market splits into {BEST_K} segments, from about {inr(seg['avg_price'].min())} (cheapest) to {inr(seg['avg_price'].max())} (most expensive) on average.")

## 9. Evaluation

In [ ]:
rows = []
for name, s in scores.items():
    for split in ["validation", "test"]:
        rows.append({"model": name, "split": split, **s[split]})
eval_df = pd.DataFrame(rows)
display(eval_df.round(3))

test_eval = eval_df[eval_df["split"] == "test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=test_eval, x="model", y="rmse", ax=axes[0], color="#3b7dd8")
axes[0].set_title("Test RMSE by model (lower is better)")
axes[0].set_xlabel("")
axes[0].set_ylabel("RMSE (INR)")
axes[0].tick_params(axis="x", rotation=20)
sns.barplot(data=test_eval, x="model", y="r2", ax=axes[1], color="#4aa36c")
axes[1].set_title("Test R2 by model (higher is better)")
axes[1].set_xlabel("")
axes[1].set_ylabel("R2")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

# predicted vs actual and residuals on a test sample
pp = test_pred.select("final_price", "pred_price").sample(False, min(1.0, 5000 / max(test.count(), 1)), SEED).toPandas()
pp["pct_error"] = (pp["pred_price"] - pp["final_price"]) / pp["final_price"] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(pp["final_price"], pp["pred_price"], s=6, alpha=0.4)
lims = [pp[["final_price", "pred_price"]].min().min(), pp[["final_price", "pred_price"]].max().max()]
axes[0].plot(lims, lims, color="red", lw=1)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_title("Predicted vs actual price (test sample)")
axes[0].set_xlabel("Actual price (INR)")
axes[0].set_ylabel("Predicted price (INR)")
sns.histplot(pp["pct_error"].clip(-50, 50), bins=60, ax=axes[1], color="#8a5fd1")
axes[1].set_title("Prediction error (%)")
axes[1].set_xlabel("(predicted - actual) / actual, %")
axes[1].set_ylabel("Cars")
plt.tight_layout()
plt.show()
print(f"Insight: the median absolute error is {pp['pct_error'].abs().median():.1f}% of the price; {100 * (pp['pct_error'].abs() < 10).mean():.0f}% of predictions are within 10%.")

In [ ]:
# Feature importance with real feature names
def get_feature_names(model, cat_cols, num_cols):
    names = []
    for i, c in enumerate(cat_cols):
        stage = model.stages[i]
        try:
            labels = list(stage.labelsArray[0])
        except Exception:
            labels = list(stage.labels)
        names += [f"{c}={l}" for l in labels]
    return names + list(num_cols)

final_stage = FINAL_MODEL.stages[-1]
if hasattr(final_stage, "featureImportances"):
    importance = final_stage.featureImportances.toArray()
    imp_label = "Importance"
else:
    importance = np.abs(final_stage.coefficients.toArray())
    imp_label = "|Coefficient| (scaled features)"

names = get_feature_names(FINAL_MODEL, CAT_COLS, BASE_NUM + COND_NUM)
if len(names) != len(importance):
    names = [f"feature_{i}" for i in range(len(importance))]
imp = pd.DataFrame({"feature": names, "importance": importance})
imp["group"] = imp["feature"].str.split("=").str[0]
imp_group = imp.groupby("group")["importance"].sum().sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
imp.sort_values("importance", ascending=False).head(15).iloc[::-1].plot.barh(x="feature", y="importance", ax=axes[0], legend=False, color="#3b7dd8")
axes[0].set_title("Top 15 individual features")
axes[0].set_xlabel(imp_label)
axes[0].set_ylabel("")
imp_group.iloc[::-1].plot.barh(ax=axes[1], color="#4aa36c")
axes[1].set_title("Top 15 features (categories combined)")
axes[1].set_xlabel(imp_label)
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()
print("Insight: the most influential features overall are:", ", ".join(imp_group.index[:5]) + ".")

## 10. Save the model and metadata
`models/model_metadata.json` holds everything the Streamlit app needs (dropdown options, slider ranges, typical specs per model, scoring weights and metrics), so retraining on new data updates the app without any code change.

In [ ]:
MODEL_PATH = MODELS_DIR / "price_model"
try:
    FINAL_MODEL.write().overwrite().save(str(MODEL_PATH))
    print("Model saved to", MODEL_PATH)
except Exception as e:
    raise RuntimeError("Could not save the model. On Windows this usually means winutils.exe/hadoop.dll are missing (see the note at the top of the notebook), then restart the kernel. Original error: " + str(e)[:600])

# ---- values derived from the data
owner_order = [r["owner"] for r in df.groupBy("owner").agg(F.avg("price").alias("p")).orderBy(F.desc("p")).collect()]

def distinct_sorted(col):
    return sorted(r[0] for r in df.select(col).distinct().collect())

brand_models = {}
for r in df.select("brand", "model").distinct().collect():
    brand_models.setdefault(r["brand"], []).append(r["model"])
brand_models = {b: sorted(m) for b, m in sorted(brand_models.items())}

spec_pdf = (df.groupBy("brand", "model", "fuel")
              .agg(F.collect_set("engine").alias("engines"), F.percentile_approx("power", 0.5).alias("power"),
                   F.percentile_approx("mileage", 0.5).alias("mileage"), F.percentile_approx("seats", 0.5).alias("seats"))
              .toPandas())
model_specs = {}
for r in spec_pdf.itertuples(index=False):
    model_specs.setdefault(r.brand, {}).setdefault(r.model, {})[r.fuel] = {
        "engine": sorted(int(e) for e in r.engines), "power": float(r.power),
        "mileage": float(r.mileage), "seats": int(round(r.seats))}

ranges = {}
for c, src in [("year", "year"), ("km_driven", "km_driven"), ("engine", "engine"), ("power", "power"),
               ("mileage", "mileage"), ("seats", "seats"), ("price", "final_price")]:
    mn, mx, md_ = df.select(F.min(src), F.max(src), F.percentile_approx(src, 0.5)).first()
    ranges[c] = {"min": float(mn), "max": float(mx), "median": float(md_)}

tier_counts = pd.Series(list(BRAND_TIER_MAP.values())).value_counts()
default_tier = tier_counts.index[0]

metadata = {
    "trained_at": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    "trained_year": CURRENT_YEAR,
    "spark_version": spark.version,
    "rows_used": int(N_ENRICHED),
    "target": "log(final_price); app applies exp() to the prediction",
    "best_model": BEST_NAME,
    "best_params": best_params,
    "input_columns": ALL_INPUTS,
    "cat_features": CAT_COLS, "base_numeric_features": BASE_NUM, "condition_features": COND_NUM,
    "parts": PARTS, "components": COMPONENTS,
    "part_weights": SETTINGS["PART_WEIGHTS"], "component_weights": SETTINGS["COMPONENT_WEIGHTS"],
    "choices": {"brand": sorted(brand_models), "location": distinct_sorted("location"),
                "fuel": distinct_sorted("fuel"), "transmission": distinct_sorted("transmission"), "owner": owner_order},
    "brand_models": brand_models,
    "model_specs": model_specs,
    "ranges": ranges,
    "brand_tier_map": BRAND_TIER_MAP,
    "default_brand_tier": default_tier,
    "grade_rule": {**SETTINGS["GRADE_RULE"], "damage_max": DAMAGE_MAX,
                   "cutoffs": GRADE_CUTOFFS, "labels": SETTINGS["GRADE_LABELS"]},
    "deal_tolerance": SETTINGS["DEAL_TOLERANCE"],
    "deductions_assumed": SETTINGS["DEDUCTIONS"],
    "metrics": {"validation": final_val, "test": final_test, "ablation_test_without_condition": abl_test,
                "condition_classifier": {"accuracy": acc, "f1": f1}, "kmeans_best_k": BEST_K},
}

def _to_py(o):
    if hasattr(o, "item"):
        return o.item()
    return str(o)

(MODELS_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2, default=_to_py), encoding="utf-8")
print("Metadata saved to", MODELS_DIR / "model_metadata.json")

## 11. Key insights, conclusion and limitations

In [ ]:
acc_r = q("SELECT accident_history, AVG(price_ratio) AS r FROM cars GROUP BY accident_history").set_index("accident_history")["r"]
part_rank = []
for part in PARTS:
    t = q(f"SELECT {part}_damage AS lvl, AVG(price_ratio) AS r FROM cars GROUP BY {part}_damage").set_index("lvl")["r"]
    if 0 in t.index and 2 in t.index:
        part_rank.append((part, 100 * (1 - t[2] / t[0])))
part_rank.sort(key=lambda x: -x[1])

print("KEY INSIGHTS (computed from this run)")
print(f"- Best model: {BEST_NAME} (tuned) with test R2 = {final_test['r2']:.3f} and median error of about {pp['pct_error'].abs().median():.1f}% of the price.")
print(f"- Condition features cut the test RMSE by {gain:.1f}% compared with a model that only knows age, km, fuel, owners and specs.")
print(f"- An accident lowers the price by about {100 * (1 - acc_r[1] / acc_r[0]):.1f}% on average (simulated assumption).")
print("- Damage that costs the most when major: " + ", ".join(f"{p} ({v:.1f}%)" for p, v in part_rank[:3]) + ".")
print(f"- The market splits into {BEST_K} natural segments (K-Means, silhouette-selected).")
print()
print("LIMITATIONS")
print("- Accident, damage, working-condition, insurance and service-history columns and the price deductions are SIMULATED assumptions, so the model learns those rules rather than real market behaviour.")
print("- The listing data itself should be validated against a real dataset before any real-world use.")
print("- Spark runs in local mode here; the same code scales to a cluster by changing the master and paths.")

In [ ]:
print("RUN SUMMARY")
print(f"Started (UTC): {RUN_STARTED:%Y-%m-%d %H:%M:%S} | finished (UTC): {dt.datetime.now(dt.timezone.utc):%Y-%m-%d %H:%M:%S}")
print(f"Spark {spark.version} | cores={CORES} | driver memory={DRIVER_MEM_GB}g | sample fraction={SETTINGS['SAMPLE_FRACTION']}")
print(f"Rows: raw={N_RAW:,} | after cleaning={N_CLEAN:,} | modelled={N_ENRICHED:,}")
print(f"Best model: {BEST_NAME} {best_params}")
print(f"Test metrics: RMSE={final_test['rmse']:,.0f} | MAE={final_test['mae']:,.0f} | R2={final_test['r2']:.4f}")
print("Timings (s):", TIMINGS)
print("Artifacts:", MODEL_PATH, "and", MODELS_DIR / "model_metadata.json")
print("Next step: streamlit run app.py")